In [6]:
import pandas as pd
import numpy as np
import cv2
import os

# Images

In [2]:
def extract_frames(video_path, output_dir, video_name, fps=30):
    # Mở video
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    # Tính toán khoảng cách frame cần lấy
    frame_interval = int(video_fps / fps) if video_fps > fps else 1

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Lấy frame theo interval
        if frame_count % frame_interval == 0:
            saved_count += 1
            frame_filename = os.path.join(output_dir, f"{video_name}_{saved_count}.jpg")
            cv2.imwrite(frame_filename, frame)

        frame_count += 1

    cap.release()
    print(f"Đã lưu {saved_count} frames vào {output_dir}")

In [2]:
INPUT_DIR = r"Data\wide_view\videos"
OUTPUT_DIR = "Data/Detection/images"

ALL_VIDEOS = sorted(os.listdir(INPUT_DIR))
TRAIN_VIDEOS = ALL_VIDEOS[:18]
VAL_VIDEOS = ALL_VIDEOS[18:24]
TEST_VIDEOS = ALL_VIDEOS[-6:]
FPS = 25
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
def split_data(subset, selected):
    print("=======================================================")
    print(f"Splitting for {subset}......")
    DIR = os.path.join(OUTPUT_DIR, subset)
    os.makedirs(DIR, exist_ok=True)
    for vid in selected:
        video_file = os.path.join(INPUT_DIR, vid)
        video_name = vid.split('.')[0]
        extract_frames(video_file, DIR, video_name, FPS)
    print(f"Splitted for {subset} with {len(selected)} video.")

In [ ]:
split_data("train", TRAIN_VIDEOS)
split_data("val", VAL_VIDEOS)
split_data("test", TEST_VIDEOS)

Splitting for val......


Đã lưu 750 frames vào Data/Detection/images\val
Đã lưu 750 frames vào Data/Detection/images\val
Đã lưu 750 frames vào Data/Detection/images\val
Đã lưu 750 frames vào Data/Detection/images\val
Đã lưu 750 frames vào Data/Detection/images\val
Đã lưu 750 frames vào Data/Detection/images\val
Splitted for val with 6 video.
Splitting for test......
Đã lưu 750 frames vào Data/Detection/images\test
Đã lưu 750 frames vào Data/Detection/images\test
Đã lưu 750 frames vào Data/Detection/images\test
Đã lưu 750 frames vào Data/Detection/images\test
Đã lưu 750 frames vào Data/Detection/images\test
Đã lưu 750 frames vào Data/Detection/images\test
Splitted for test with 6 video.


# Labels

In [3]:
WIDTH = 6500
HEIGHT = 1000
SOURCE_DIR = r"Data\wide_view\annotations"
DEST_DIR = r"Data\Detection\labels"

os.makedirs(DEST_DIR, exist_ok=True)

In [4]:
def build_object_list(raw):
    # Lấy dòng đầu tiên - Player ID và Ball
    n_cols = raw.iloc[0].tolist() #93 phần tử -> 92 cột và 1 từ "PlayerID"
    
    # Mỗi phần tử trong objects ứng với 1 đối tượng thật -> 23 phần tử
    objects = []
    col = 1 #Bỏ cột 0 vì cột 0 là cột frames
    while col + 3 < len(n_cols): # Mỗi object kéo dài 4 cột gồm height, left, top, width
        team = str(n_cols[col]).strip().upper()
        if team != "BALL":
            objects.append((1, col, col + 1, col + 2, col + 3))
        col += 4
    return objects

In [7]:
def convert(csv_path, out_dir, video_name):
    os.makedirs(out_dir, exist_ok=True)
    raw = pd.read_csv(csv_path, header=None)
    objects = build_object_list(raw)
    data_rows = raw.iloc[4:]  # Bỏ qua 4 dòng headers
    n_written = 0
 
    # Duyệt qua từng frame, mỗi frame lưu 1 file txt
    for _, row in data_rows.iterrows():
        vals = row.tolist()
        try:
            frame_id = int(float(str(vals[0]).strip()))
        except ValueError:
            continue  
        
        # Duyệt qua từng object trong frame -> 23 objects -> 1 file txt có 23 dòng
        lines = []
        for (class_id, h_col, l_col, t_col, w_col) in objects:
            try:
                h = float(vals[h_col])
                left = float(vals[l_col])
                top = float(vals[t_col])
                w = float(vals[w_col])
            except (ValueError, IndexError):
                continue
 
            if any(np.isnan(v) for v in [h, left, top, w]):
                continue
 
            # YOLO: center x, center y, width, height — all normalized
            cx = (left + w / 2) / WIDTH
            cy = (top + h / 2) / HEIGHT
            nw = w / WIDTH
            nh = h / HEIGHT
 
            # Đảm bảo trong khoảng [0, 1]
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            nw = max(0.0, min(1.0, nw))
            nh = max(0.0, min(1.0, nh))
 
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
 
        out_path = os.path.join(out_dir, f"{video_name}_{frame_id}.txt")
        with open(out_path, "w") as f:
            f.write("\n".join(lines) + "\n")
 
        n_written += 1
 
    print(f"Đã tạo {n_written} files annotation.")

In [10]:
def split_anno(subset, selected):
    print("=======================================================")
    print(f"Splitting for {subset}......")
    DIR = os.path.join(DEST_DIR, subset)
    os.makedirs(DIR, exist_ok=True)
    for vid in selected:
        video_name = vid.split('.')[0]
        vid = video_name + ".csv"
        src_file = os.path.join(SOURCE_DIR, vid)
        convert(src_file, DIR, video_name)
    print(f"Splitted for {subset} with {len(selected)} video.")

In [11]:
split_anno("train", TRAIN_VIDEOS)
split_anno("val", VAL_VIDEOS)
split_anno("test", TEST_VIDEOS)

Splitting for train......
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Splitted for train with 18 video.
Splitting for val......
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Splitted for val with 6 video.
Splitting for test......
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files annotation.
Đã tạo 750 files an